# word2vec 속도 개선
전 장에서 만든 모델은 말뭉치에 포함된 어휘의 수가 많아지면 계산량이 커진다. 그래서 계산 시간을 줄이기 위한 방법을 찾아보자.

## word2vec 속도 개선 1
기존의 예제에서는 7개의 단어의 말뭉치를 사용하였지만 100만 개의 단어의 말뭉치를 사용하면 문제가 2가지가 발생한다. 
* 입력층의 원핫 벡터와 입력 측 가중치 행렬의 곱 계산 => Embedding 계층 도입
* 은닉층과 출력 측 가중치 행렬의 곱 및 softmax 계층의 계산 => 네거티브 샘플링이라는 새로운 손실 함수 도입

### Embedding 계층
우리는 원핫벡터와 가중치 행렬을 곱하여 얻는 것이라고는 행렬의 특정 행을 추출하는 것뿐이다. 그래서 원핫 벡터 변환과 행렬 곱 연산은 필요가 없다. 그러면 단어 Id에 해당하는 행을 추출만 하는 계층을 만들어야 하는데 이를 Embedding 계층이라고 한다. 특정 행을 추출하는 방법은 어렵지 않다. 어떤 행렬 A이 있을 때 A[n] 처럼 원하는 행을 명시하기만 하면 된다. 그리고 역전파는 순전파가 가중치 W의 특정 행만 추출하는 것이였기 때문에 앞 층에서 전해진 기울기를 다음 층으로 그래도 흘려주기만 하면 전해진 앞 층 기울기를 특정 행에 설정한다. 그리고 그렇게 구현하면 주석처럼 하게 되는데 이때에는 중복된 원소가 있을 때 덮어씌워지는 일이 일어나기 때문에 더하는 것으로 해결한다.

In [3]:
import numpy as np

class Embedding:
    def __init__(self, W):
        self.params = [W]
        self.grads = [np.zeros_like(W)]
        self.idx = None

    def forward(self, idx):
        W, = self.params
        self.idx = idx
        out = W[idx]
        return out

    def backward(self, dout):
        dW, = self.grads
        dW[...] = 0
        # dW[self.idx] = dout
        np.add.at(dW, self.idx, dout)
        return None

## word2vec 속도 개선 2

### 은닉층 이후 계산의 문제점
은닉층의 벡터 크기가 100이고 가중치 행렬의 크기가 100 x 100만 이므로 상당히 계산이 오래 걸리고 역전파 때도 같은 계산을 수행해서 가볍게 해야하고 softmax에서도 계산량이 증가하게 된다. 그래서 네거티브 샘플링 기법을 차용하는데 이 기법은 다중 분류를 이진 분류로 근사하는 것이다. 즉 질문의 답이 yes or no로 나오게 바꾸는 신경망을 고안하는 것이다. 그것은 바로 은닉층과 출력 측 가중치 행렬의 내적에서 어떤 단어에 해당하는 열만 추출하고 추출한 벡터와 은닉층 뉴런과의 내적을 계산 하면 되는 것이다. 그 결과를 시그노이드 함수에 적용해 확률로 변환하고 손실을 구하기 위해서 교차 엔트로피 오차를 사용한다. 이것과 Embbedding을 합친 Embedding Dot을 구현하면 아래와 같다.

In [4]:
import sys
sys.path.append('..')
from np import np
from layers import Embedding, SigmoidWithLoss

class EmbeddingDot:
    def __init__(self, W):
        self.embed = Embedding(W)
        self.params = self.embed.params
        self.grads = self.embed.grads
        self.cache = None

    def forward(self, h, idx):
        target_W = self.embed.forward(idx)
        out = np.sum(target_W * h, axis=1)

        self.cache = (h, target_W)
        return out

    def backward(self, dout):
        h, target_W = self.cache
        dout = dout.reshape(dout.shape[0], 1)

        dtarget_W = dout * h
        self.embed.backward(dtarget_W)
        dh = dout * target_W
        return dh



### 네거티브 샘플링
앞 신경망에서는 부정적인 예를 입력하였을 때의 결과가 확실하지 않다. 즉 부정적인 예를 입력하였을 때 시그모이드 계층의 출력이 0에 가깝게 만들고 싶은 것이다. 그래서 모든 부정적인 예를 대상으로 이진 분류 학습을 시키면 좋겠지만 어휘 수가 많으면 연산이 감당할 수 없이 많아진다. 그래서 부정적인 예 몇 개를 선택하는 샘플링을 진행한다. 그래서 정리하면 긍정적이 예를 타깃으로 할 때의 손실을 구하고 그와 동시에 부정적인 예 몇 개를 샘플링하여 그 예에 대해서도 손실을 구한다. 그리고 각각으 데이터의 손실을 더한 값을 최종 손실로 한다.  
  
그러면 부정적인 예를 어떻게 샘플링할까? 말뭉치의 통계 데이터를 기반으로 이루어진다. 그래서 자주 등장하는 단어를 많이 추출하면 된다. 그래서 단어별 출현 횟수를 바탕으로 확률분포를 구한 다음에 샘플링을 수행하면 된다. 그럼 하번 구현 해보자.

In [5]:
import collections
from np import np

class UnigramSampler:
    def __init__(self, corpus, power, sample_size):
        self.sample_size = sample_size
        self.vocab_size = None
        self.word_p = None

        counts = collections.Counter()
        for word_id in corpus:
            counts[word_id] += 1

        vocab_size = len(counts)
        self.vocab_size = vocab_size

        self.word_p = np.zeros(vocab_size)
        for i in range(vocab_size):
            self.word_p[i] = counts[i]

        self.word_p = np.power(self.word_p, power)
        self.word_p /= np.sum(self.word_p)

    def get_negative_sample(self, target):
        batch_size = target.shape[0]

        import config
        if not config.GPU:
            negative_sample = np.zeros((batch_size, self.sample_size), dtype=np.int32)

            for i in range(batch_size):
                p = self.word_p.copy()
                target_idx = target[i]
                p[target_idx] = 0
                p /= p.sum()
                negative_sample[i, :] = np.random.choice(self.vocab_size, size=self.sample_size, replace=False, p=p)
        else:
            # GPU(cupy）로 계산할 때는 속도를 우선한다.
            # 부정적 예에 타깃이 포함될 수 있다.
            negative_sample = np.random.choice(self.vocab_size, size=(batch_size, self.sample_size),
                                               replace=True, p=self.word_p)

        return negative_sample


class NegativeSamplingLoss:
    def __init__(self, W, corpus, power=0.75, sample_size=5):
        self.sample_size = sample_size
        self.sampler = UnigramSampler(corpus, power, sample_size)
        self.loss_layers = [SigmoidWithLoss() for _ in range(sample_size + 1)]
        self.embed_dot_layers = [EmbeddingDot(W) for _ in range(sample_size + 1)]

        self.params, self.grads = [], []
        for layer in self.embed_dot_layers:
            self.params += layer.params
            self.grads += layer.grads

    def forward(self, h, target):
        batch_size = target.shape[0]
        negative_sample = self.sampler.get_negative_sample(target)

        # 긍정적 예 순전파
        score = self.embed_dot_layers[0].forward(h, target)
        correct_label = np.ones(batch_size, dtype=np.int32)
        loss = self.loss_layers[0].forward(score, correct_label)

        # 부정적 예 순전파
        negative_label = np.zeros(batch_size, dtype=np.int32)
        for i in range(self.sample_size):
            negative_target = negative_sample[:, i]
            score = self.embed_dot_layers[1 + i].forward(h, negative_target)
            loss += self.loss_layers[1 + i].forward(score, negative_label)

        return loss

    def backward(self, dout=1):
        dh = 0
        for l0, l1 in zip(self.loss_layers, self.embed_dot_layers):
            dscore = l0.backward(dout)
            dh += l1.backward(dscore)

        return dh

## 개선판 word2vec 학습
PTB 데이터 셋을 사용하여 학습 시켜보자. 2_4_1_train.py 참고
모델 평가도 진행하자.

In [1]:
import sys
sys.path.append('..')
from util import most_similar, analogy
import pickle


pkl_file = 'cbow_params.pkl'
# pkl_file = 'skipgram_params.pkl'

with open(pkl_file, 'rb') as f:
    params = pickle.load(f)
    word_vecs = params['word_vecs']
    word_to_id = params['word_to_id']
    id_to_word = params['id_to_word']

# 가장 비슷한(most similar) 단어 뽑기
querys = ['you', 'year', 'car', 'toyota']
for query in querys:
    most_similar(query, word_to_id, id_to_word, word_vecs, top=5)

# 유추(analogy) 작업
print('-'*50)
analogy('king', 'man', 'queen',  word_to_id, id_to_word, word_vecs)
analogy('take', 'took', 'go',  word_to_id, id_to_word, word_vecs)
analogy('car', 'cars', 'child',  word_to_id, id_to_word, word_vecs)
analogy('good', 'better', 'bad',  word_to_id, id_to_word, word_vecs)

------------------------------------------------------------
                       GPU Mode (cupy)
------------------------------------------------------------


[query] you
 we: 0.7529296875
 i: 0.67236328125
 they: 0.61376953125
 your: 0.60498046875
 anybody: 0.59619140625

[query] year
 month: 0.85205078125
 summer: 0.77001953125
 spring: 0.76953125
 week: 0.763671875
 decade: 0.7060546875

[query] car
 truck: 0.6044921875
 auto: 0.5986328125
 window: 0.59765625
 luxury: 0.5791015625
 merkur: 0.578125

[query] toyota
 seita: 0.65283203125
 nissan: 0.6181640625
 honda: 0.607421875
 motor: 0.6044921875
 marathon: 0.6005859375
--------------------------------------------------

[analogy] king:man = queen:?
 a.m: 6.21875
 woman: 5.2265625
 mother: 4.875
 answers: 4.85546875
 naczelnik: 4.7890625

[analogy] take:took = go:?
 were: 4.4609375
 eurodollars: 4.3515625
 came: 4.29296875
 went: 4.171875
 're: 4.00390625

[analogy] car:cars = child:?
 a.m: 6.30859375
 rape: 5.953125
 children: